# GRPO Adaptation for Vision-Language Models with an SFT Warm Start

This is a curated static coursework artifact. It preserves selected custom implementation excerpts for inspection; setup, authentication, data acquisition, training calls, saved outputs, and execution history have been removed. The original experiments were not rerun during curation.


## Structured-answer rewards

The following excerpt parses the expected MILP summary format and computes formatting and partial-credit rewards. It is retained as static source only.


In [ ]:
import re

_MILP_LINE = re.compile(
    r"(Variables:\s*\d+,\s*Constraints:\s*\d+,\s*Density:\s*\w+,\s*"
    r"Binary:\s*[\d.]+%,\s*Integer:\s*[\d.]+%,\s*Objective:\s*\w+)",
    re.IGNORECASE | re.DOTALL,
)


def extract_answer(text: str) -> str | None:
    """Extract the final structured MILP answer from a completion."""
    text = (text or "").strip()
    match = re.search(r"Answer:\s*(.+)", text, re.IGNORECASE | re.DOTALL)
    if match:
        rest = match.group(1).strip()
        for line in rest.splitlines():
            line = line.strip()
            if line:
                return line
        return rest
    match = _MILP_LINE.search(text)
    return match.group(1).strip() if match else None


def parse_milp_fields(text: str | None):
    if not text:
        return None
    match = re.search(
        r"Variables:\s*(\d+),\s*Constraints:\s*(\d+),\s*Density:\s*(\w+),\s*"
        r"Binary:\s*([\d.]+%),\s*Integer:\s*([\d.]+%),\s*Objective:\s*(\w+)",
        text,
        re.IGNORECASE | re.DOTALL,
    )
    return match.groups() if match else None


def accuracy_reward(completions: list, answer: list, **kwargs) -> list:
    """Score exact or six-field partial agreement with a reference answer."""
    def _looks_milp(value: str) -> bool:
        return "Variables:" in value and "Constraints:" in value and "Objective:" in value

    def _milp_field_score(prediction: str, reference: str) -> float:
        predicted_fields = parse_milp_fields(prediction)
        reference_fields = parse_milp_fields(reference)
        if not predicted_fields or not reference_fields:
            return 0.0
        return sum(
            left.strip().lower() == right.strip().lower()
            for left, right in zip(predicted_fields, reference_fields)
        ) / 6.0

    rewards = []
    for completion, ground_truth in zip(completions, answer):
        prediction = extract_answer(completion[0]["content"])
        if prediction is None:
            rewards.append(0.0)
        elif prediction == ground_truth:
            rewards.append(1.0)
        elif _looks_milp(ground_truth) and _looks_milp(prediction):
            rewards.append(_milp_field_score(prediction, ground_truth))
        else:
            rewards.append(0.0)
    return rewards


def format_reward(completions: list, **kwargs) -> list:
    """Reward completions containing a parseable structured MILP answer."""
    return [
        1.0 if extract_answer(completion[0]["content"]) is not None else 0.0
        for completion in completions
    ]


## SFT data collation

This excerpt retains the custom collator used for answer-only loss masking in the SFT stage. Model loading, adapter persistence, and external data paths are intentionally omitted.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Any, List
import json
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForImageTextToText, AutoProcessor, Trainer, TrainingArguments

class VLSFTCollator:
    """Labels default: loss only on assistant tokens (answer line), not on user/image/prompt."""

    processor: Any
    max_seq_len: int
    mask_prompt_loss: bool = True

    def _prompt_prefix_len(self, user_messages: list, text_full: str, img) -> int:
        """Token count shared at sequence start between prompt-only and full chat encoding."""
        text_prompt = self.processor.apply_chat_template(
            user_messages, tokenize=False, add_generation_prompt=True
        )
        ep = self.processor(
            text=[text_prompt],
            images=[img],
            return_tensors="pt",
            truncation=True,
            max_length=self.max_seq_len,
        )
        ef = self.processor(
            text=[text_full],
            images=[img],
            return_tensors="pt",
            truncation=True,
            max_length=self.max_seq_len,
        )
        lp = int(ep["attention_mask"][0].sum())
        lf = int(ef["attention_mask"][0].sum())
        pa = ep["input_ids"][0, :lp]
        pb = ef["input_ids"][0, :lf]
        m = min(lp, lf)
        k = 0
        for i in range(m):
            if pa[i] != pb[i]:
                break
            k += 1
        return k

    def __call__(self, features: List[dict]) -> dict[str, torch.Tensor]:
        texts: list[str] = []
        images: list = []
        prompt_lens: list[int] = []

        for ex in features:
            img = ex["image"]
            if getattr(img, "mode", None) != "RGB":
                img = img.convert("RGB")
            user_content = ex["prompt"][0]["content"]
            text_part = next(c["text"] for c in user_content if c["type"] == "text")
            user_messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img},
                        {"type": "text", "text": text_part},
                    ],
                }
            ]
            messages = [
                *user_messages,
                {"role": "assistant", "content": [{"type": "text", "text": ex["answer"]}]},
            ]
            t_full = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            if self.mask_prompt_loss:
                prompt_lens.append(self._prompt_prefix_len(user_messages, t_full, img))
            texts.append(t_full)
            images.append(img)

        batch = self.processor(
            text=texts,
            images=images,
            padding=True,
            truncation=True,
            max_length=self.max_seq_len,
            return_tensors="pt",
        )
        labels = batch["input_ids"].clone()
        labels[batch["attention_mask"] == 0] = -100
        if self.mask_prompt_loss and prompt_lens:
            for i, pl in enumerate(prompt_lens):
                seq = int(batch["attention_mask"][i].sum().item())
                eff = min(pl, seq)
                labels[i, :eff] = -100
        batch["labels"] = labels
        for im in images:
            try:
                im.close()
            except Exception:
                pass
        return batch


## Archived workflow

The original coursework used a LoRA-based SFT stage and could initialize GRPO from that adapter or from a fresh LoRA adapter. The archived results and limitations are documented in the project README; no training or evaluation invocation is retained in this notebook.
